# Data Ingestion and Transformation Demo Using Medallion Architecture with PySpark DataFrames

This notebook shows you:
- how to load raw employee data from an S3 bucket into `bronze` layer tables
- how to apply various transformations on the raw data and save the cleaned data into `silver` layer tables
- how to prepare aggregated data for business use cases and save them to `gold` layer tables

### Prerequisites:

- Create the `databricks_training_employees` catalog with `bronze`, `silver` and `gold` schemas.
- Prepare data in S3 and set up the connection from Databricks:
    - Connect to your S3 bucket, by creating a storage credential and an external location for S3 from the Databricks Catalog Explorer (follow the steps from [here](https://docs.databricks.com/aws/en/connect/unity-catalog/cloud-storage/s3/s3-external-location-manual)).
    - Upload the necessary CSV files (from `sample-data/employee_data`) to your S3 bucket and modify the `base_file_path` accordingly.
- Alternatively, you can upload the CSV files from the `sample-data/employee_data` folder to a Unity Catalog volume and load them into DataFrames from there.

In [0]:
# Define the base file path from AWS S3
base_file_path = "s3://szk-databricks-training-demo-s3/employee_data/"

# Read each file into a named DataFrame using the base file path
employees_branch_a = spark.read.format("csv").option("header", "true").load(base_file_path + "employees_branch_a.csv")
employees_branch_b = spark.read.format("csv").option("header", "true").load(base_file_path + "employees_branch_b.csv")
job_history_branch_a = spark.read.format("csv").option("header", "true").load(base_file_path + "job_history_branch_a.csv")
job_history_branch_b = spark.read.format("csv").option("header", "true").load(base_file_path + "job_history_branch_b.csv")

In [0]:
# # Alternatively, you can upload the CSV files from the sample-data/employee_data folder to the `databricks_training_employees.raw.employees` Unity Catalog volume and load them from there:

# catalog = "databricks_training_employees"
# schema = "raw"
# volume = "employees"
# path_volume = "/Volumes/" + catalog + "/" + schema + "/" + volume

# # Read each file into a named DataFrame from the Unity Catalog volume
# employees_branch_a = spark.read.csv(f"{path_volume}/employees_branch_a.csv", header=True)
# employees_branch_b = spark.read.csv(f"{path_volume}/employees_branch_b.csv", header=True)
# job_history_branch_a = spark.read.csv(f"{path_volume}/job_history_branch_a.csv", header=True)
# job_history_branch_b = spark.read.csv(f"{path_volume}/job_history_branch_a.csv", header=True)

In [0]:
employees_branch_a.write.mode("overwrite").saveAsTable("databricks_training_employees.00_bronze.employees_branch_a")
employees_branch_b.write.mode("overwrite").saveAsTable("databricks_training_employees.00_bronze.employees_branch_b")
job_history_branch_a.write.mode("overwrite").saveAsTable("databricks_training_employees.00_bronze.job_history_branch_a")
job_history_branch_b.write.mode("overwrite").saveAsTable("databricks_training_employees.00_bronze.job_history_branch_b")

In [0]:
# Display a few rows from each DataFrame
displayHTML("<h2><b>employees_branch_a:</b></h2>")
display(employees_branch_a.limit(5))

displayHTML("<h2><b>employees_branch_b:</b></h2>")
display(employees_branch_b.limit(5))

displayHTML("<h2><b>job_history_branch_a:</b></h2>")
display(job_history_branch_a.limit(5))

displayHTML("<h2><b>job_history_branch_b:</b></h2>")
display(job_history_branch_b.limit(5))

In [0]:
from pyspark.sql import functions as F

In [0]:
# Typecasting employees DataFrame
employees_branch_a = employees_branch_a.withColumn("age", F.col("age").cast("int")) \
    .withColumn("salary", F.col("salary").cast("double")) \
    .withColumn("join_date", F.col("join_date").cast("date"))

employees_branch_b = employees_branch_b.withColumn("age", F.col("age").cast("int")) \
    .withColumn("salary", F.col("salary").cast("double")) \
    .withColumn("join_date", F.col("join_date").cast("date"))

# Typecasting job_history DataFrame
job_history_branch_a = job_history_branch_a.withColumn("start_date", F.col("start_date").cast("date")) \
    .withColumn("end_date", F.col("end_date").cast("date"))

job_history_branch_b = job_history_branch_b.withColumn("start_date", F.col("start_date").cast("date")) \
    .withColumn("end_date", F.col("end_date").cast("date"))


In [0]:
# Union the employees datasets and job history datasets
employees_union = employees_branch_a.union(employees_branch_b)
job_history_union = job_history_branch_a.union(job_history_branch_b)

In [0]:
# Join the unioned employees with the unioned job history
joined_data = employees_union.join(job_history_union, on="employee_id", how="inner")

In [0]:
display(joined_data)

In [0]:
# Filter rows with salary more than 50000
filtered_data = joined_data.filter(F.col("salary") > 50000)
display(filtered_data)

In [0]:
# Select only relevant columns using 'select' or 'drop'
selected_data = filtered_data.select("employee_id", "name", "age", "salary", "department", "join_date", "gender", "role", "start_date", "end_date")

display(selected_data)

# Alternatively, use 'drop' to exclude a column (in this case, 'drop' may be more readable)
# selected_data = filtered_data.drop("branch")

In [0]:
# Sort based on employee_id and start_date (desc)
sorted_data = selected_data.orderBy(["employee_id", "start_date"], ascending=[True, False])
display(sorted_data)

In [0]:
# Deduplicate to keep the latest active role for each employee
deduped_data = sorted_data.dropDuplicates(["employee_id"])
display(deduped_data)

In [0]:
# Calculate years_in_service using join_date and current date
data_with_years = deduped_data.withColumn(
    "years_in_service", 
    F.floor(F.datediff(F.current_date(), F.to_date(F.col("join_date"))) / 365)
)
display(data_with_years)

In [0]:
# Drop rows where the role is Null
cleaned_data = data_with_years.dropna(subset=["role"])

# Replace Null values in age with 0
cleaned_data = cleaned_data.fillna({"age": 0})

display(cleaned_data)

In [0]:
# Replace 'M' with 'Male', 'F' with 'Female', and any other value with 'Other' in the 'gender' column
cleaned_data = cleaned_data.withColumn(
    "gender", 
    F.when(F.col("gender") == "M", "Male")
     .when(F.col("gender") == "F", "Female")
     .otherwise("Other")
)

display(cleaned_data)

In [0]:
cleaned_data.write.mode("overwrite").saveAsTable("databricks_training_employees.01_silver.employees_cleaned")

In [0]:
# Calculate the average, minimum, and maximum salary for each role
aggregated_salary_by_role = cleaned_data.groupBy("role").agg(
    F.avg("salary").alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary")
)
displayHTML("<h2><b>Salary summary for each employee role:</b></h2>")
display(aggregated_salary_by_role)

# Calculate the average, minimum, and maximum salary by department
aggregated_salary_by_department = cleaned_data.groupBy("department").agg(
    F.avg("salary").alias("avg_salary"),
    F.min("salary").alias("min_salary"),
    F.max("salary").alias("max_salary")
)
displayHTML("<h2><b>Salary summary by department:</b></h2>")
display(aggregated_salary_by_department)

department_summary = cleaned_data.groupBy("department").agg(
    F.count("employee_id").alias("employee_count"),
    F.sum(F.when(F.col("gender") == "Female", 1).otherwise(0)).alias("female_count"),
    F.sum(F.when(F.col("gender") == "Male", 1).otherwise(0)).alias("male_count"),
    F.avg("years_in_service").alias("avg_years_in_service"),
    F.avg("age").alias("avg_age")
)
displayHTML("<h2><b>Department summary:</b></h2>")
display(department_summary)


Databricks visualization. Run in Databricks to view.

In [0]:
import matplotlib.pyplot as plt

pdf = aggregated_salary_by_role.limit(5).toPandas()
pdf.plot(kind="bar", x="role", y="avg_salary", color="skyblue", figsize=(10, 5))

plt.xlabel("Role")
plt.ylabel("Average Salary")
plt.title("Average Salary by Role")
plt.xticks(rotation=45)
plt.show()

In [0]:
aggregated_salary_by_role.write.mode("overwrite") \
    .saveAsTable("databricks_training_employees.02_gold.employee_role_salary_summary")

aggregated_salary_by_department.write.mode("overwrite") \
    .saveAsTable("databricks_training_employees.02_gold.department_salary_summary")

department_summary.write.mode("overwrite") \
    .saveAsTable("databricks_training_employees.02_gold.department_summary")